# 03 · Modelo de conversión montaje + referencia (10-10 bipolar → 10-20 monopolar)

**Rama `explore/topomap-refs`** · entrenamiento **dentro del notebook** (TF/Keras,
CPU lineal) con datos reales cacheados.

## La física del problema (por qué no "monopolar Cz" a secas)

Del bipolar **no se puede recuperar el nivel absoluto** del unipolar: si
`X_bip = X_u @ M_b` con `M_b` de suma de columnas 0, entonces `X_u + c(t)·1`
produce el mismo bipolar para cualquier `c(t)`. El unipolar monopolar **Cz**
tiene ese nivel como parte de la señal → "10-10 bipolar → 10-20 monopolar **Cz**"
es mal-posed. Se fija la salida a la referencia de superficie **`average_all`**
del notebook 02, que es lo físicamente recuperable.

## La ruta es UNA sola matriz

`X_bip = X_u10 @ M_b`, `X_u20 = X_u10 @ P` (spline) e `X20avg = X_u20 @ M_avg20`
(NO=20 average_all). Componiendo, la conversión **montaje+referencia** se
colapsa en un único operador lineal:

\[ W_{an} = \text{pinv}(M_b)\,P\,M_{avg20}, \qquad X_{bip}\,W_{an} = X_{20avg} \]

(el modo que `pinv(M_b)` no recupera —constante por instante— lo anula
`M_avg20`, de columnas con suma 0). Se comparan tres rutas en test:

1. **línea base analítica** (paso a paso) → techo del spline (`VE ≈ 0.97`);
2. **mínimos cuadrados cerrados** sobre datos → la ruta es *recuperable
   exactamente* (`VE = 1.0`);
3. **red lineal entrenada con Adam** (precondicionada) → `VE ≈ 0.99`,
   venciendo a la línea base.

## 0. Entorno y configuración

In [1]:
import os, sys, json
import numpy as np

ROOT = os.getcwd()
# ascenso hasta la raíz del repo (donde vive src/),
# robusto a la profundidad de notebooks/exploraciones/topomap_refs/
while not os.path.isdir(os.path.join(ROOT, "src")):
    ROOT = os.path.dirname(ROOT)
os.chdir(ROOT)
for p in (os.path.join(ROOT, "src"),):
    if p not in sys.path:
        sys.path.insert(0, p)

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
keras.utils.set_random_seed(0)
tf.get_logger().setLevel("ERROR")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

from eeg_transform.nb import load_experiment, multiconfig_data
from eeg_transform.mapping import (select_subset, STANDARD_10_20_19,
                                   spherical_spline_matrix, scalp_grid_matrix)
from eeg_transform.experiments.multi import MONTAGE_10_10_39
from eeg_transform.references import build_reference_matrix

OUT = os.path.join(ROOT, "runs", "topomap_refs")
FIG = os.path.join(OUT, "03_figs")
os.makedirs(FIG, exist_ok=True)

def ve(a, b):
    a0 = a - a.mean(0, keepdims=True)
    b0 = b - b.mean(0, keepdims=True)
    return float(1 - ((a0 - b0) ** 2).sum() / (b0 ** 2).sum())

def rmse_uV(a, b): return float(np.sqrt(((a - b) ** 2).mean()) * 1e6)

print("ROOT:", ROOT, "| TF", tf.__version__)

I0000 00:00:1790019531.715067  141736 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1790019531.715391  141736 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1790019531.755178  141736 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1790019532.623498  141736 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1790019532.623715  141736 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


ROOT: /home/aess/Proyectos/universal-eeg-transformer | TF 2.21.0


## 1. Datos, montajes y construcción de la tarea

In [2]:
cfg, ds = load_experiment("config/universal_refs.yaml")
mc = multiconfig_data(cfg, ds)["canonical"]
n1010, p1010, i1010 = select_subset(ds.ch_names, ds.ch_positions, keep=MONTAGE_10_10_39)
n1020, p1020, i1020 = select_subset(ds.ch_names, ds.ch_positions, keep=STANDARD_10_20_19)
print("10-10:", len(n1010), "canales | 10-20:", len(n1020), "canales")

Mb = build_reference_matrix("bipolar", len(n1010), positions=np.asarray(p1010))
P = spherical_spline_matrix(np.asarray(p1010), np.asarray(p1020),
                            smoothness=cfg.mapping.smoothness)
M20, ms20, _ = scalp_grid_matrix(np.asarray(p1020), grid_px=48)
w20 = M20[ms20].mean(0)
Mavg20 = np.eye(len(n1020)) - w20[:, None]

def build(split):
    Xu = mc.refs[split]["unipolar"]
    Xbip = Xu[:, i1010] @ Mb          # (T, 42) pares bipolares de la 10-10
    X20avg = Xu[:, i1020] @ Mavg20    # (T, 19) 10-20 monopolar average_all
    return Xbip, X20avg

Xtr, Ytr = build("train"); Xva, Yva = build("val"); Xte, Yte = build("test")
print("formas: X", Xtr.shape, "| Y", Ytr.shape)
print("M_b: suma de columnas max |.| = %.1e (gauge bipolar)" % float(np.abs(Mb.sum(0)).max()))
W_an = np.linalg.pinv(Mb, rcond=1e-10) @ P @ Mavg20
print("W_an analítico (pinv·spline·avg) → VE train=%.4f test=%.4f (techo del spline)"
      % (ve(Xtr @ W_an, Ytr), ve(Xte @ W_an, Yte)))

10-10: 42 canales | 10-20: 19 canales
formas: X (3000, 42) | Y (3000, 19)
M_b: suma de columnas max |.| = 0.0e+00 (gauge bipolar)
W_an analítico (pinv·spline·avg) → VE train=0.9711 test=0.9704 (techo del spline)


## 2. Línea base analítica paso a paso (bipolar → mínimo-norma → spline → average_all)

In [3]:
def baseline(Xbip, true):
    U10 = Xbip @ np.linalg.pinv(Mb, rcond=1e-10)
    return (U10 @ P) @ Mavg20

Yb_tr = baseline(Xtr, Ytr); Yb_va = baseline(Xva, Yva); Yb_te = baseline(Xte, Yte)
for nm, Yb, Y in [("train", Yb_tr, Ytr), ("val", Yb_va, Yva), ("test", Yb_te, Yte)]:
    print("baseline %s  VE=%.4f  RMSE=%.2f µV" % (nm, ve(Yb, Y), rmse_uV(Yb, Y)))

baseline train  VE=0.9711  RMSE=3.66 µV
baseline val  VE=0.9735  RMSE=3.62 µV
baseline test  VE=0.9704  RMSE=3.56 µV


## 3. Aprendizaje de la ruta con datos

### 3.1 Mínimos cuadrados cerrados (LS sobre 3000 muestras, SVD completo)

Con solo un pase se recupera *exactamente* la ruta (VE = 1.0 en test): la
elización a través de `41→19` determinista y las 19 salidas viven en el
subespacio de las 42 entradas bipolares.

### 3.2 Red lineal entrenada con Adam (TF/Keras)

El aprendizaje por gradiente colapsa por mal condicionamiento: los 42 canales
bipolares están altamente correlacionados (espectro singular con `cond ≈ 1e16`,
mínimo valor singular ~1e-14). Se **precondiciona** la entrada (whitening PCA
retiene el 99.9 % de la varianza, k≈39 ejes) y la capa lineal converge a
`VE ≈ 0.99`, superando a la línea base analítica (0.97), que paga el
alisado del spline.

In [4]:
# 3.1 LS cerrada (SVD de la entrada estandarizada por canal)
mx, sx = Xtr.mean(0), Xtr.std(0)
my, sy = Ytr.mean(0), Ytr.std(0)
A = (Xtr - mx) / sx; B = (Ytr - my) / sy
U, S_, Vt = np.linalg.svd(A, full_matrices=False)
W_ls = Vt.T @ ((S_ / (S_ ** 2 + 1e-12))[:, None] * (U.T @ B))
print("espectro singular de la entrada: max=%.3g  mín=%.2e  cond=%g"
      % (S_[0], S_[-1], S_[0] / S_[-1]))

def predict_ls(X):
    return ((X - mx) / sx @ W_ls) * sy + my

Yls_tr, Yls_te = predict_ls(Xtr), predict_ls(Xte)
print("LS cerrada         train VE=%.6f  RMSE=%.3f µV" % (ve(Yls_tr, Ytr), rmse_uV(Yls_tr, Ytr)))
print("LS cerrada         test  VE=%.6f  RMSE=%.3f µV" % (ve(Yls_te, Yte), rmse_uV(Yls_te, Yte)))

espectro singular de la entrada: max=134  mín=1.54e-14  cond=8.68353e+15
LS cerrada         train VE=1.000000  RMSE=0.000 µV
LS cerrada         test  VE=1.000000  RMSE=0.000 µV


In [5]:
# 3.2 Adam con whitening PCA (retiene varianza acumulada 99.9 %)
var = S_ ** 2; cum = np.cumsum(var) / var.sum()
k = int(np.where(cum > 0.999)[0][0]) + 1
print("rango efectivo (99.9 % varianza): k =", k, "de", A.shape[1])
Vk = Vt[:k].T
Aw = A @ Vk; Avw = (Xva - mx) / sx @ Vk; Atw = (Xte - mx) / sx @ Vk

model = keras.Sequential([keras.layers.Input((k,)),
                          keras.layers.Dense(Ytr.shape[1], activation=None)])
model.compile(optimizer=keras.optimizers.Adam(1e-2), loss="mse")
hist = model.fit(Aw, B, validation_data=(Avw, (Yva - my) / sy), epochs=300,
                 batch_size=3000, verbose=0,
                 callbacks=[keras.callbacks.EarlyStopping(patience=60, restore_best_weights=True),
                            keras.callbacks.ReduceLROnPlateau(patience=30, factor=0.5, min_lr=1e-5)])
ep = len(hist.history["loss"])
Wk = model.layers[0].get_weights()[0]
Yp_te = (Atw @ Wk) * sy + my
Yp_tr = (Aw @ Wk) * sy + my
print("Adam whitening    epochs=%d  val_loss=%.5f" % (ep, hist.history["val_loss"][-1]))
print("Adam whitening    train VE=%.4f RMSE=%.2f µV | test VE=%.4f RMSE=%.2f µV"
      % (ve(Yp_tr, Ytr), rmse_uV(Yp_tr, Ytr), ve(Yp_te, Yte), rmse_uV(Yp_te, Yte)))

rango efectivo (99.9 % varianza): k = 39 de 42


E0000 00:00:1790019533.923779  141736 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Adam whitening    epochs=300  val_loss=0.01229
Adam whitening    train VE=0.9928 RMSE=1.83 µV | test VE=0.9915 RMSE=1.91 µV


## 4. Evaluación en test (verdad vs modelo-vs-línea base)

In [6]:
route = {"baseline": Yb_te, "LS": Yls_te, "modelo_Adam": Yp_te}
rows = []
for nm, Yp in route.items():
    per = [ve(Yp[:, j], Yte[:, j]) for j in range(Yte.shape[1])]
    rows.append((nm, ve(Yp, Yte), rmse_uV(Yp, Yte), float(np.median(per)),
                 float(np.min(per)), n1020[int(np.argmin(per))]))
for nm, v, rm, pm, pmin, ch in rows:
    print("%-12s  VE=%.4f  RMSE=%.2f µV  VE/canal med=%.4f min=%.4f (%s)"
          % (nm, v, rm, pm, pmin, ch))
results = {"baseline": {"VE": rows[0][1], "RMSE_uV": rows[0][2]},
           "ls_cerrada": {"VE": rows[1][1], "RMSE_uV": rows[1][2]},
           "modelo_adam": {"VE": rows[2][1], "RMSE_uV": rows[2][2],
                           "epocas": ep, "k_precond": k,
                           "parametros": k * 19 + 19},
           "per_canal_modelo": {n1020[j]: float(ve(Yp_te[:, j], Yte[:, j]))
                                for j in range(19)},
           "cond_entrada": float(S_[0] / S_[-1])}

t = 200
fig, axes = plt.subplots(1, 3, figsize=(13.8, 4.6))
def _t(Xt, ax, title):
    Mm, m, _ = scalp_grid_matrix(np.asarray(p1020), grid_px=48)
    img = (np.asarray(Xt).reshape(1, -1) @ Mm.T).reshape(48, 48)
    img = np.where(m.reshape(48, 48), img, np.nan)
    vlim = float(np.nanmax(np.abs(img)))
    im = ax.imshow(img, cmap="RdBu_r", vmin=-vlim, vmax=vlim, origin="lower", interpolation="bilinear")
    ax.set_xticks([]); ax.set_yticks([]); ax.set_title(title, fontsize=9)
    return im
for ax, Xt, tt in zip(axes, [Yte, Yp_te, Yb_te],
                      ["10-20 average_all (verdad)", "modelo Adam", "baseline analítica"]):
    im = _t(Xt[t], ax, f"{tt} @ t={t}")
    fig.colorbar(im, ax=ax, shrink=0.8, pad=0.02)
fig.suptitle("Conversión 10-10 bipolar → 10-20 monopolar (average_all)", y=1.02)
fig.tight_layout(); fig.savefig(os.path.join(FIG, "modelo_topomapas.png"), dpi=110); plt.close(fig)

fig, ax = plt.subplots(figsize=(8.4, 4))
cj = np.arange(19)
ax.plot(cj, [ve(Yp_te[:, j], Yte[:, j]) for j in cj], "o-", label="modelo Adam")
ax.plot(cj, [ve(Yb_te[:, j], Yte[:, j]) for j in cj], "s--", label="baseline")
ax.plot(cj, [ve(Yls_te[:, j], Yte[:, j]) for j in cj], "^", label="LS cerrada")
ax.axhline(1.0, color="k", lw=0.6)
ax.set_xticks(cj); ax.set_xticklabels(n1020, rotation=90, fontsize=7)
ax.set_ylabel("VE por canal (test)"); ax.legend(); ax.grid(alpha=0.3)
ax.set_title("VE por canal: modelo vs baseline vs LS")
fig.tight_layout(); fig.savefig(os.path.join(FIG, "modelo_ve_canal.png"), dpi=110); plt.close(fig)

with open(os.path.join(OUT, "03_metrics.json"), "w") as fh:
    json.dump(results, fh, indent=2, default=float)
print("métricas ->", os.path.join(OUT, "03_metrics.json"))

baseline      VE=0.9704  RMSE=3.56 µV  VE/canal med=0.9726 min=0.8620 (F3)
LS            VE=1.0000  RMSE=0.00 µV  VE/canal med=1.0000 min=1.0000 (Fp1)
modelo_Adam   VE=0.9915  RMSE=1.91 µV  VE/canal med=0.9874 min=0.9639 (C3)


métricas -> /home/aess/Proyectos/universal-eeg-transformer/runs/topomap_refs/03_metrics.json


## 5. Resumen

* Sin tocar `src/`, un **único peso lineal** aprendido convierte **10-10
  bipolar → 10-20 monopolar (`average_all`)** combinando montaje (spline) y
  referencia (promedio de superficie) en una vía `X_bip @ W`.
* La ruta es *determinista*: LS cerrada la recupera exacta (VE = 1.0). Adam
  (SGD) exige precondicionamiento (whitening PCA) por el **mal
  condicionamiento** de los canales bipolares (cond ≈ 1e16); condicionada,
  alcanza VE ≈ 0.99 y **supera** a la línea base analítica (0.97, techo del
  alisado del spline).
* Lección física: del bipolar el **nivel monopolar Cz es inobservable**
  (gauge); `average_all` resuelve la indeterminación. Esto conecta con el
  autoencoder lineal universal del repo: `A_{s→d} = W^{enc}_s W^{dec}_d`
  aprende exactamente este tipo de rutas.

In [7]:
print("RESUMEN")
print("  baseline     test VE=%.4f RMSE=%.2f µV" % (results["baseline"]["VE"], results["baseline"]["RMSE_uV"]))
print("  LS cerrada   test VE=%.4f RMSE=%.2f µV" % (results["ls_cerrada"]["VE"], results["ls_cerrada"]["RMSE_uV"]))
print("  modelo Adam  test VE=%.4f RMSE=%.2f µV (k=%d, %d épocas)"
      % (results["modelo_adam"]["VE"], results["modelo_adam"]["RMSE_uV"],
         results["modelo_adam"]["k_precond"], results["modelo_adam"]["epocas"]))

RESUMEN
  baseline     test VE=0.9704 RMSE=3.56 µV
  LS cerrada   test VE=1.0000 RMSE=0.00 µV
  modelo Adam  test VE=0.9915 RMSE=1.91 µV (k=39, 300 épocas)
